# Pose Transfer — mannequin -> chibi animada (EXPERIMENTAL)

Rota **(A)** do ADR-002: cada frame e gerado por IA, guiado por um frame de
mannequin renderizado no Blender.

**Isto nao e a estrategia de producao.** O ADR-002 escolheu *rig cutout* e
descartou (A) por **drift e flicker**. Este notebook existe para decidir com
evidencia, conforme o ADR-007.

## O que avaliar

O modo de falha previsto e **cintilacao entre frames**, e ele e **invisivel
frame a frame**. Julgue o **GIF em loop**, nas escalas reais (64/96/128 px),
nunca a grade de frames a 1024 px.

Decisao de aprovar ou reprovar e **humana**.

## Insumos (todos por upload — nada pesado no Git)

| Insumo | O que e |
|---|---|
| frames do mannequin | `walk_00.png` … `walk_05.png`, ou um sheet unico |
| chibi master | a personagem ja gerada (ex.: `run_003_output.png`) |
| referencia opcional | face/outfit, se quiser reforcar identidade |


In [ ]:
#@title 0. Painel { display-mode: "form" }
#@markdown ### Identificacao
CHARACTER_ID = "waifu_001"  #@param {type:"string"}
ANIM_NAME = "walk"  #@param ["walk", "idle", "attack", "hurt", "custom"]
ANIM_CUSTOM = ""  #@param {type:"string"}

#@markdown ### Frames
#@markdown Deixe `0` para usar o padrao de `config/project.yaml`
#@markdown (walk=6, idle=4). O numero real vem dos arquivos enviados.
FRAME_COUNT = 0  #@param {type:"integer"}

#@markdown ### Como o mannequin foi enviado
MANNEQUIN_MODE = "frames_separados"  #@param ["frames_separados", "sheet_unico"]
#@markdown Se `sheet_unico`, informe a grade (colunas x linhas):
SHEET_COLS = 6  #@param {type:"integer"}
SHEET_ROWS = 1  #@param {type:"integer"}

#@markdown ### Como a pose entra no modelo
#@markdown Todos os modos usam o MESMO checkpoint. Muda so a imagem de pose
#@markdown (e, no ultimo, a LoRA). Compare um fator de cada vez.
POSE_INPUT_MODE = "render_direto"  #@param ["render_direto", "dwpose_skeleton", "depth_map", "depth_map_lora"]
#@markdown  - `render_direto`: o render cinza do mannequin, como veio do Blender.
#@markdown  - `dwpose_skeleton`: esqueleto DWPose. So articulacao, sem volume
#@markdown    de adulto — tende a contaminar menos a proporcao chibi.
#@markdown  - `depth_map`: mapa de profundidade. Carrega volume 3D.
#@markdown  - `depth_map_lora`: depth + RefControl LoRA (Apache-2.0).
#@markdown    ATENCAO: a LoRA declara base NAO destilada; usamos a destilada.
LORA_STRENGTH = 0.9  #@param {type:"slider", min:0.0, max:1.2, step:0.05}

#@markdown ### Geracao
SEED_MODE = "fixa_em_todos_os_frames"  #@param ["fixa_em_todos_os_frames", "incremental_por_frame"]
SEED = 42  #@param {type:"integer"}
#@markdown `fixa` maximiza consistencia entre frames — e o padrao recomendado
#@markdown para animacao. `incremental` existe so para diagnostico.

STEPS = 20  #@param {type:"integer"}
CFG = 2.5  #@param {type:"number"}

#@markdown ### Prompt (generico e reutilizavel — sem traços da personagem)
PROMPT_MODE = "generico"  #@param ["generico", "manual"]
PROMPT_MANUAL = ""  #@param {type:"string"}
NEGATIVE_CUSTOM = ""  #@param {type:"string"}

#@markdown ### Saida
GERAR_GIF = True  #@param {type:"boolean"}
ESCALAS_AVALIACAO = "64,96,128"  #@param {type:"string"}

import pathlib

ANIM = (ANIM_CUSTOM or "").strip() if ANIM_NAME == "custom" else ANIM_NAME
assert ANIM, "ANIM_CUSTOM vazio com ANIM_NAME='custom'."
assert "/" not in ANIM and "/" not in CHARACTER_ID, "sem barras nos nomes."
assert CHARACTER_ID.strip(), "CHARACTER_ID vazio."
if PROMPT_MODE == "manual":
    assert PROMPT_MANUAL.strip(), "PROMPT_MODE='manual' exige PROMPT_MANUAL."

# O prompt NAO descreve a personagem: identidade vem da imagem de referencia.
# Ele descreve apenas a TAREFA (repor a mesma personagem noutra pose).
PROMPT_GENERICO = (
    "Redraw the same chibi character from the reference in the pose shown by "
    "the grey mannequin, keeping the identity, outfit and accessories "
    "unchanged. Full body, same art style, same colors, clean cel shading, "
    "clean lineart, plain background."
)

# A LoRA so existe para o modo que a usa; o trigger word e obrigatorio nela.
USA_LORA = POSE_INPUT_MODE == "depth_map_lora"
USA_PREPROCESSADOR = POSE_INPUT_MODE != "render_direto"
if USA_LORA:
    PROMPT_GENERICO = "refcontrol " + PROMPT_GENERICO
assert 0.0 <= LORA_STRENGTH <= 1.2

REPO = pathlib.Path("/content/ChibiCreate")
REPO_BRANCH = "arena/01a07ece-chibicreate"
BASE = pathlib.Path("/content/pose_transfer")
UP_POSE = BASE / "upload_mannequin"
UP_CHAR = BASE / "upload_character"
FRAMES_IN = BASE / "pose_frames"
FRAMES_OUT = BASE / "out_frames"
for d in (UP_POSE, UP_CHAR, FRAMES_IN, FRAMES_OUT):
    d.mkdir(parents=True, exist_ok=True)

ESCALAS = [int(x) for x in ESCALAS_AVALIACAO.split(",") if x.strip()]

print("personagem :", CHARACTER_ID)
print("animacao   :", ANIM)
print("seed       :", SEED, "|", SEED_MODE)
print("escalas    :", ESCALAS)
print()
print("modo pose :", POSE_INPUT_MODE, "| lora:", USA_LORA,
      ("(peso %.2f)" % LORA_STRENGTH) if USA_LORA else "")
print()
print("EXPERIMENTAL (ADR-007) — nao e a rota de producao.")
print("Avalie o GIF em loop, nao os frames isolados.")


In [ ]:
#@title 1. Repositorio + ComfyUI { display-mode: "form" }
import subprocess, sys

if REPO.exists():
    subprocess.run(["git","-C",str(REPO),"fetch","-q","origin",REPO_BRANCH], check=True)
    subprocess.run(["git","-C",str(REPO),"checkout","-q","-B",REPO_BRANCH,
                    "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git","clone","-q","--branch",REPO_BRANCH,
                    "https://github.com/BloomRX/ChibiCreate.git", str(REPO)],
                   check=True)

SCRIPTS = REPO / "scripts"
if not (SCRIPTS / "chibi" / "flow01.py").exists():
    raise SystemExit(
        f"BLOCKED — {SCRIPTS}/chibi nao existe. A branch '{REPO_BRANCH}' foi "
        "baixada? A main do repositorio so tem o README.")
sys.path.insert(0, str(SCRIPTS))

print("commit:", subprocess.run(["git","-C",str(REPO),"rev-parse","HEAD"],
      capture_output=True, text=True).stdout.strip())

# O ComfyUI e o motor: sem ele a celula de execucao morre com
# "can't open file '/content/ComfyUI/main.py'".
COMFY = pathlib.Path("/content/ComfyUI")
if not (COMFY / "main.py").exists():
    print("clonando ComfyUI...")
    subprocess.run(["git","clone","-q",
                    "https://github.com/comfyanonymous/ComfyUI.git",
                    str(COMFY)], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r",
                    str(COMFY / "requirements.txt")], check=False)
subprocess.run([sys.executable,"-m","pip","install","-q",
                "pyyaml","pillow","numpy","huggingface_hub"], check=False)

if not (COMFY / "main.py").exists():
    raise SystemExit(f"BLOCKED — {COMFY}/main.py nao existe: o clone falhou.")
COMFY_COMMIT = subprocess.run(["git","-C",str(COMFY),"rev-parse","HEAD"],
                              capture_output=True, text=True).stdout.strip()
print("ComfyUI:", COMFY_COMMIT[:10])
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Preprocessadores (DWPose/depth). Custom node autorizado e PINADO por commit
# — main flutuante quebraria a auditoria. Registrado em models.lock.yaml.
AUX_COMMIT = "59b1fc411ede8623b2997855b8018f0b3b6cf49f"
AUX = COMFY / "custom_nodes" / "comfyui_controlnet_aux"
if USA_PREPROCESSADOR:
    if not AUX.exists():
        print("clonando comfyui_controlnet_aux...")
        subprocess.run(["git","clone","-q",
                        "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
                        str(AUX)], check=True)
    subprocess.run(["git","-C",str(AUX),"checkout","-q",AUX_COMMIT], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r",
                    str(AUX / "requirements.txt")], check=False)
    got = subprocess.run(["git","-C",str(AUX),"rev-parse","HEAD"],
                         capture_output=True, text=True).stdout.strip()
    if got != AUX_COMMIT:
        raise SystemExit(f"BLOCKED — aux em {got}, esperado {AUX_COMMIT}")
    print("controlnet_aux pinado:", got[:10])
else:
    print("modo render_direto: preprocessador nao e necessario")


In [ ]:
#@title 1b. Ambiente e GPU (detectado, nao presumido) { display-mode: "form" }
import sys, json

WEIGHTS_GB = 7.75      # flux-2-klein-4b.safetensors
MIN_VRAM_GB = 13.0     # model card oficial: '~13GB VRAM'

try:
    import torch
except ImportError:
    torch = None

print('=' * 62)
print('AMBIENTE COLAB — detectado, nao presumido')
print('=' * 62)
print('Python :', sys.version.split()[0])
print('Torch  :', torch.__version__ if torch else 'ausente')

if torch is None or not torch.cuda.is_available():
    print('CUDA   : INDISPONIVEL')
    print('COLAB_GPU_INSUFFICIENT — nenhuma GPU CUDA.')
    print('Runtime -> Alterar tipo de ambiente de execucao -> GPU')
    raise SystemExit('FASE 3B permanece BLOCKED')

props = torch.cuda.get_device_properties(0)
total_gb = props.total_memory / 1024 ** 3
free_gb = torch.cuda.mem_get_info()[0] / 1024 ** 3

GPU_INFO = {
    'name': props.name,
    'vram_total_gb': round(total_gb, 2),
    'vram_free_gb': round(free_gb, 2),
    'cuda': torch.version.cuda,
    'capability': '{}.{}'.format(props.major, props.minor),
    'torch': torch.__version__,
    'python': sys.version.split()[0],
    'bf16_supported': props.major >= 8,
}
print('GPU    :', GPU_INFO['name'])
print('VRAM   : {:.2f} GB total / {:.2f} GB livre'.format(total_gb, free_gb))
print('CUDA   :', GPU_INFO['cuda'], '| capability', GPU_INFO['capability'])
print('Necessario : ~{:.0f} GB'.format(MIN_VRAM_GB))
print()

if not GPU_INFO['bf16_supported']:
    print('AVISO: sem bf16 nativo (capability < 8.0, ex. T4).')
    print('O ComfyUI cai para fp16. Pode funcionar; registre o resultado.')
    print()

if total_gb < MIN_VRAM_GB:
    print('=' * 62)
    print('COLAB_GPU_INSUFFICIENT')
    print('=' * 62)
    print('{} tem {:.1f} GB; o FLUX.2 klein 4B precisa de ~{:.0f} GB.'
          .format(GPU_INFO['name'], total_gb, MIN_VRAM_GB))
    print('PARE. Nao usar CPU, nao trocar de modelo, nao quantizar por conta.')
    raise SystemExit('COLAB_GPU_INSUFFICIENT')

print('GPU ADEQUADA — pode prosseguir.')
json.dump(GPU_INFO, open('/content/gpu_info.json', 'w'), indent=2)


In [ ]:
#@title 2. Upload do mannequin (frames ou sheet) { display-mode: "form" }
from PIL import Image

try:
    from google.colab import files
    print("Envie os frames do mannequin (walk_00.png ... walk_05.png)")
    print("ou um unico sheet, conforme MANNEQUIN_MODE.")
    for nome, dados in files.upload().items():
        (UP_POSE / nome).write_bytes(dados)
        print("recebido:", nome, len(dados), "bytes")
except Exception as e:
    print("[fora do Colab] copie os arquivos para", UP_POSE, "|", e)

_envio = [p for p in sorted(UP_POSE.iterdir())
          if p.suffix.lower() in (".png", ".webp")]
if not _envio:
    raise SystemExit(f"BLOCKED — nenhum PNG em {UP_POSE}.")

for p in FRAMES_IN.glob("*.png"):
    p.unlink()

if MANNEQUIN_MODE == "sheet_unico":
    if len(_envio) != 1:
        raise SystemExit(
            f"BLOCKED — MANNEQUIN_MODE='sheet_unico' mas vieram {len(_envio)} "
            "arquivos. Envie um so, ou mude para 'frames_separados'.")
    sheet = Image.open(_envio[0]).convert("RGBA")
    if sheet.width % SHEET_COLS or sheet.height % SHEET_ROWS:
        raise SystemExit(
            f"BLOCKED — sheet {sheet.size} nao divide exatamente por "
            f"{SHEET_COLS}x{SHEET_ROWS}. A grade esta errada: um corte "
            "desalinhado desloca o pivot e o sprite 'pula' no loop.")
    fw, fh = sheet.width // SHEET_COLS, sheet.height // SHEET_ROWS
    n = 0
    for r in range(SHEET_ROWS):
        for c in range(SHEET_COLS):
            sheet.crop((c * fw, r * fh, (c + 1) * fw, (r + 1) * fh)).save(
                FRAMES_IN / f"{ANIM}_{n:02d}.png")
            n += 1
    print(f"sheet {sheet.size} -> {n} frames de {fw}x{fh}")
else:
    for i, p in enumerate(_envio):
        Image.open(p).convert("RGBA").save(FRAMES_IN / f"{ANIM}_{i:02d}.png")
    print(f"{len(_envio)} frames importados")

POSE_FRAMES = sorted(FRAMES_IN.glob(f"{ANIM}_*.png"))
if FRAME_COUNT and len(POSE_FRAMES) != FRAME_COUNT:
    raise SystemExit(
        f"BLOCKED — FRAME_COUNT={FRAME_COUNT} mas ha {len(POSE_FRAMES)} frames.")
print("frames de pose:", len(POSE_FRAMES))


In [ ]:
#@title 3. Validar o mannequin (canvas, pivot, alfa, proporcao) { display-mode: "form" }
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# POSE_FRAMES e relido do disco (mesma fonte da celula 2), nao sobrescrito
# com outro significado: recomputar aqui torna a celula re-executavel sozinha.
POSE_FRAMES = sorted(FRAMES_IN.glob(f"{ANIM}_*.png"))
if not POSE_FRAMES:
    raise SystemExit("BLOCKED — rode a celula 2.")

BLOQUEIOS, AVISOS = [], []
tamanhos = {Image.open(p).size for p in POSE_FRAMES}
if len(tamanhos) != 1:
    BLOQUEIOS.append(f"canvas inconsistente entre frames: {tamanhos}")

caixas = []
for p in POSE_FRAMES:
    im = Image.open(p).convert("RGBA")
    a = np.array(im)
    alpha = a[:, :, 3]
    if alpha.max() == 0:
        BLOQUEIOS.append(f"{p.name}: totalmente transparente")
        caixas.append(None)
        continue
    if alpha.min() == 255:
        AVISOS.append(f"{p.name}: sem transparencia (fundo solido)")
        # silhueta por diferenca do canto: fundo chroma uniforme
        fundo = a[0, 0, :3]
        ocupado = (np.abs(a[:, :, :3].astype(int) - fundo).sum(2) > 30)
    else:
        ocupado = alpha > 16
    ys, xs = np.where(ocupado)
    caixas.append((xs.min(), ys.min(), xs.max(), ys.max()) if len(xs) else None)

validas = [c for c in caixas if c]
if not validas:
    BLOQUEIOS.append("nenhum frame com conteudo detectavel")
else:
    # Pivot: o chao (y maximo) tem de ser estavel entre frames.
    chao = [c[3] for c in validas]
    deriva = max(chao) - min(chao)
    alturas = [c[3] - c[1] for c in validas]
    h_med = sum(alturas) / len(alturas)
    if deriva > 0.05 * h_med:
        AVISOS.append(
            f"pivot instavel: a base varia {deriva}px entre frames "
            f"({deriva / h_med:.1%} da altura). O sprite vai 'pular' no loop. "
            "Verifique se a camera e ortografica e imovel no Blender.")
    print(f"altura media do sujeito : {h_med:.0f}px")
    print(f"deriva da base          : {deriva}px")

print()
print("PROPORCAO — [HUMAN REVIEW REQUIRED]")
print("  O mannequin do Mixamo e humano (~7 cabecas); chibi tem ~2-3.")
print("  Este notebook NAO corrige proporcao automaticamente: isso")
print("  esconderia a causa da falha. Corrija no Blender, antes do render.")
print("  Olhe os frames abaixo e decida se a silhueta ja e chibi.")

n = len(POSE_FRAMES)
fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.8))
for ax, p in zip(np.atleast_1d(axes), POSE_FRAMES):
    ax.imshow(Image.open(p).convert("RGBA"))
    ax.set_title(p.stem, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(str(BASE / "mannequin_frames.png"), dpi=80)
plt.close()
display(Image.open(BASE / "mannequin_frames.png"))

for a in AVISOS:
    print("[aviso]", a)
if BLOQUEIOS:
    for b in BLOQUEIOS:
        print("[BLOQUEIO]", b)
    raise SystemExit(f"BLOCKED — {len(BLOQUEIOS)} problema(s) no mannequin.")
print()
print("mannequin OK.")


In [ ]:
#@title 3b. Preprocessar a pose (DWPose / depth) { display-mode: "form" }
#@markdown Converte cada frame do mannequin no sinal de controle escolhido.
#@markdown Em `render_direto` nao faz nada.
import json, urllib.request, time, shutil

POSE_USADA = sorted(FRAMES_IN.glob(f"{ANIM}_*.png"))
PREPROC_INFO = {"mode": POSE_INPUT_MODE, "node": None, "commit": None}

if not USA_PREPROCESSADOR:
    print("render_direto — frames usados como vieram do Blender.")
else:
    NODE = {"dwpose_skeleton": "DWPreprocessor",
            "depth_map": "DepthAnythingV2Preprocessor",
            "depth_map_lora": "DepthAnythingV2Preprocessor"}[POSE_INPUT_MODE]

    # Nao oferecer no que o servidor nao tem: validar contra /object_info.
    with urllib.request.urlopen("http://127.0.0.1:8188/object_info",
                                timeout=30) as r:
        INFO = json.load(r)
    if NODE not in INFO:
        candidatos = sorted(k for k in INFO
                            if "Pose" in k or "Depth" in k)[:15]
        raise SystemExit(
            f"BLOCKED — o no '{NODE}' nao existe neste servidor. O custom node "
            f"foi instalado (celula 1)? Disponiveis: {candidatos}")
    PREPROC_INFO["node"] = NODE
    PREPROC_INFO["commit"] = AUX_COMMIT
    print("no de preprocessamento:", NODE)

    PRE_DIR = BASE / "pose_pre"
    PRE_DIR.mkdir(exist_ok=True)
    for f in PRE_DIR.glob("*.png"):
        f.unlink()

    COMFY_IN = COMFY / "input"
    COMFY_IN.mkdir(parents=True, exist_ok=True)
    for i, src in enumerate(POSE_USADA):
        nome = f"prep_src_{i:02d}.png"
        shutil.copy2(src, COMFY_IN / nome)
        g = {"1": {"class_type": "LoadImage",
                   "inputs": {"image": nome, "upload": "image"}},
             "2": {"class_type": NODE,
                   "inputs": {"image": ["1", 0], "resolution": 1024}},
             "3": {"class_type": "SaveImage",
                   "inputs": {"images": ["2", 0],
                              "filename_prefix": f"prep_{i:02d}"}}}
        req = urllib.request.Request(
            "http://127.0.0.1:8188/prompt",
            data=json.dumps({"prompt": g}).encode(),
            headers={"Content-Type": "application/json"})
        pid = json.load(urllib.request.urlopen(req, timeout=30))["prompt_id"]
        img = None
        for _ in range(180):
            time.sleep(2)
            with urllib.request.urlopen(
                    f"http://127.0.0.1:8188/history/{pid}", timeout=10) as r:
                h = json.load(r)
            if pid in h:
                for no in h[pid].get("outputs", {}).values():
                    for im in no.get("images", []):
                        img = im
                break
        if not img:
            raise SystemExit(
                f"BLOCKED — preprocessador nao devolveu imagem no frame {i}. "
                "Degradar em silencio aqui produziria uma comparacao falsa.")
        url = (f"http://127.0.0.1:8188/view?filename={img['filename']}"
               f"&subfolder={img.get('subfolder','')}&type={img.get('type','output')}")
        alvo = PRE_DIR / f"{ANIM}_{i:02d}.png"
        alvo.write_bytes(urllib.request.urlopen(url, timeout=60).read())
        print(f"  frame {i} -> {alvo.name}")

    POSE_USADA = sorted(PRE_DIR.glob(f"{ANIM}_*.png"))
    if len(POSE_USADA) != len(list(FRAMES_IN.glob(f"{ANIM}_*.png"))):
        raise SystemExit("BLOCKED — contagem de frames mudou no preprocessamento.")

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
    n = len(POSE_USADA)
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.8))
    for ax, q in zip(np.atleast_1d(axes), POSE_USADA):
        ax.imshow(Image.open(q))
        ax.set_title(q.stem, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(str(BASE / "pose_preprocessada.png"), dpi=80)
    plt.close()
    display(Image.open(BASE / "pose_preprocessada.png"))
    print()
    print("[HUMAN REVIEW REQUIRED] O sinal acima representa a pose que voce")
    print("  quer? Se o esqueleto estiver quebrado ou o depth chapado, o")
    print("  resultado vai herdar o defeito.")

print()
print("frames de pose em uso:", len(POSE_USADA))


In [ ]:
#@title 4. Upload da chibi (imagem principal) + referencia opcional { display-mode: "form" }
import hashlib

try:
    from google.colab import files
    print("Envie a chibi ja gerada (ex.: run_003_output.png).")
    print("Opcionalmente envie tambem 1 referencia extra (face/outfit).")
    for nome, dados in files.upload().items():
        (UP_CHAR / nome).write_bytes(dados)
        print("recebido:", nome, len(dados), "bytes")
except Exception as e:
    print("[fora do Colab] copie a arte para", UP_CHAR, "|", e)

_c = [p for p in sorted(UP_CHAR.iterdir())
      if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp")]
if not _c:
    raise SystemExit(f"BLOCKED — nenhuma imagem em {UP_CHAR}.")

CHIBI = max(_c, key=lambda p: Image.open(p).size[0] * Image.open(p).size[1])
REF_EXTRA = next((p for p in _c if p != CHIBI), None)

def _sha(p):
    return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()

def _pixel_sha(p):
    im = Image.open(p).convert("RGBA")
    return hashlib.sha256(im.tobytes()).hexdigest()

CHIBI_SHA = _sha(CHIBI)
CHIBI_PIXEL_SHA = _pixel_sha(CHIBI)
_im = Image.open(CHIBI)
print()
print("chibi       :", CHIBI.name, _im.size, _im.mode)
print("sha256      :", CHIBI_SHA)
print("pixel sha256:", CHIBI_PIXEL_SHA)
print("ref extra   :", REF_EXTRA.name if REF_EXTRA else "(nenhuma)")
display(_im.copy().resize((320, int(_im.height * 320 / _im.width))))


In [ ]:
#@title 5. Baixar os pesos do FLUX.2 klein { display-mode: "form" }
from huggingface_hub import hf_hub_download
import pathlib, shutil

# HF_REPO, nao REPO: a celula 1 define REPO como o caminho do clone do
# ChibiCreate e a celula 6 usa esse valor como cwd. Reaproveitar o nome aqui
# fazia o subprocess tentar entrar em 'Comfy-Org/vae-...'.
M = pathlib.Path('/content/ComfyUI/models')
HF_REPO = 'Comfy-Org/vae-text-encorder-for-flux-klein-4b'
REV  = '5f526678002e43af5551dadb73ce2e8c91b43afe'

DOWNLOADS = [
    ('split_files/diffusion_models/flux-2-klein-4b.safetensors',
     M / 'diffusion_models',
     'ec3d4e733a771f61c052fb4856c48b336c55eaf2c65487c2a1faeb9bbda7a343'),
    ('split_files/text_encoders/qwen_3_4b.safetensors',
     M / 'text_encoders',
     '6c671498573ac2f7a5501502ccce8d2b08ea6ca2f661c458e708f36b36edfc5a'),
    # Alternativa fp4 para GPU apertada — troque a linha acima por esta:
    # ('split_files/text_encoders/qwen_3_4b_fp4_flux2.safetensors',
    #  M / 'text_encoders',
    #  '3eab03a77adb0ee5304a4e677d5c10ac22f9049c1d7c894adca4f8bb39206ca8'),
    ('split_files/vae/flux2-vae.safetensors',
     M / 'vae',
     '868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3'),
]

MODEL_RECORD = []
for remote, dest, expected in DOWNLOADS:
    dest.mkdir(parents=True, exist_ok=True)
    fname = remote.split('/')[-1]
    target = dest / fname
    if target.exists():
        print('ja existe:', fname)
    else:
        print('baixando :', fname)
        got = hf_hub_download(repo_id=HF_REPO, revision=REV, filename=remote)
        shutil.copy(got, target)
    MODEL_RECORD.append({'file': fname, 'repo': HF_REPO, 'revision': REV,
                         'license': 'Apache-2.0',
                         'size_bytes': target.stat().st_size,
                         'sha256_expected': expected})
!df -h /content | tail -1


In [ ]:
#@title 5b. Conferir SHA256 dos pesos { display-mode: "form" }
import hashlib, pathlib, json

def sha256_of(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

ok = True
for rec in MODEL_RECORD:
    hit = list(pathlib.Path('/content/ComfyUI/models').rglob(rec['file']))[0]
    actual = sha256_of(hit)
    rec['sha256_actual'] = actual
    rec['sha256_match'] = actual == rec['sha256_expected']
    ok &= rec['sha256_match']
    print(('OK   ' if rec['sha256_match'] else 'FALHA'), rec['file'])

json.dump(MODEL_RECORD, open('/content/model_record.json', 'w'), indent=2)
if not ok:
    raise SystemExit('SHA256 divergente — PARE e reporte.')
print('Pesos conferem com models.lock.yaml.')


In [ ]:
#@title 5c. Subir o ComfyUI { display-mode: "form" }
import subprocess, time, urllib.request, json, sys, pathlib

if not pathlib.Path('/content/ComfyUI/main.py').exists():
    raise SystemExit(
        'BLOCKED — /content/ComfyUI/main.py nao existe. Rode a celula 1, '
        'que clona o ComfyUI, antes desta.')


LOG = open('/content/comfyui.log', 'w')
proc = subprocess.Popen([sys.executable, 'main.py', '--listen', '127.0.0.1',
                         '--port', '8188'],
                        cwd='/content/ComfyUI', stdout=LOG,
                        stderr=subprocess.STDOUT)

print('subindo ComfyUI...')
for i in range(120):
    time.sleep(5)
    try:
        with urllib.request.urlopen('http://127.0.0.1:8188/system_stats',
                                    timeout=5) as r:
            stats = json.load(r)
        print('no ar apos ~{}s'.format((i + 1) * 5))
        break
    except Exception:
        if proc.poll() is not None:
            print(open('/content/comfyui.log').read()[-3000:])
            raise SystemExit('ComfyUI morreu ao iniciar')
else:
    print(open('/content/comfyui.log').read()[-3000:])
    raise SystemExit('ComfyUI nao respondeu em 10 min')

print(json.dumps(stats.get('system', {}), indent=2))


In [ ]:
#@title 6. Prompt e workflow resolvido { display-mode: "form" }
import json, sys

sys.path.insert(0, str(REPO / "scripts"))
from chibi.model_registry import termos_especificos_no_prompt

PROMPT = PROMPT_GENERICO if PROMPT_MODE == "generico" else PROMPT_MANUAL.strip()
PROMPT_SOURCE = ("preset:pose_transfer_generic_v1" if PROMPT_MODE == "generico"
                 else "manual")
CHARACTER_SPECIFIC = PROMPT_MODE != "generico"

_achados = termos_especificos_no_prompt(PROMPT)
if _achados and not CHARACTER_SPECIFIC:
    raise SystemExit(
        f"BLOCKED — prompt marcado como generico mas cita {_achados}. "
        "O prompt-base precisa servir a 100+ personagens: a identidade vem "
        "da imagem de referencia, nunca do texto.")
if _achados:
    print("[aviso] prompt especifico:", _achados)
    print("        registrado como character_specific_prompt=true")

# v2 = v1 + LoRA no caminho do MODEL. So no modo que a usa.
WF_PATH = REPO / ("workflows/experimental/pose_transfer/v2.json" if USA_LORA
                  else "workflows/experimental/pose_transfer/v1.json")
WF = json.loads(WF_PATH.read_text())
print("workflow:", WF_PATH.relative_to(REPO))
print("slots   :", json.dumps(WF["_slots"], ensure_ascii=False))
print()
print("cadeia de referencia (a ORDEM importa — o 1o ReferenceLatent e a pose):")
for k in sorted((k for k in WF if not k.startswith("_")), key=int):
    ct = WF[k]["class_type"]
    if ct in ("LoadImage", "ReferenceLatent", "KSampler"):
        print(f"  {k:>3} {ct:<18} {json.dumps(WF[k]['inputs'], ensure_ascii=False)[:90]}")
print()
LORA_FILE = None
if USA_LORA:
    from huggingface_hub import hf_hub_download
    import hashlib, shutil
    LORA_REPO = "thedeoxen/refcontrol-FLUX.2-klein-4B-reference-depth-lora"
    LORA_REV = "0ae1ef7f9acc4e55ec2237360943c3c3032d3583"
    LORA_SHA = "65ec4c71fa7538b2201481928609a5836773f0dc06a4041b2d28abd05826c401"
    LORA_FILE = "flux2_klein_4b_refcontrol_depth.safetensors"
    destino = COMFY / "models" / "loras" / LORA_FILE
    destino.parent.mkdir(parents=True, exist_ok=True)
    if not destino.exists():
        got = hf_hub_download(repo_id=LORA_REPO, revision=LORA_REV,
                              filename=LORA_FILE)
        shutil.copy(got, destino)
    achado = hashlib.sha256(destino.read_bytes()).hexdigest()
    if achado != LORA_SHA:
        raise SystemExit(
            f"BLOCKED — sha256 da LoRA nao confere.\n  esperado {LORA_SHA}"
            f"\n  obtido   {achado}")
    print()
    print("LoRA:", LORA_FILE, "| sha256 confere | Apache-2.0")
    print("peso:", LORA_STRENGTH, "| trigger 'refcontrol' no prompt")
    print()
    print("[TEST REQUIRED] base_mismatch: a LoRA declara klein-base-4B (NAO")
    print("  destilada) e o pipeline usa a destilada. Aderencia fraca pode")
    print("  ser ISSO, e nao a tecnica. Nao concluir sem considerar.")

print("prompt:", PROMPT)
print()
print("NOTA: ReferenceLatent NAO e ControlNet. A pose INFLUENCIA a geracao,")
print("      nao a trava. Aderencia frouxa e resultado esperado, e deve ser")
print("      reportada — nao compensada em silencio.")


In [ ]:
#@title 7. Gerar os frames (uma execucao por frame) { display-mode: "form" }
import subprocess, time, shutil, json, urllib.request

if not (REPO / "scripts" / "chibi" / "cli.py").exists():
    raise SystemExit(
        f"BLOCKED — REPO={REPO!r} nao aponta para o clone do ChibiCreate. "
        "Alguma celula sobrescreveu a variavel: rode a celula 1 de novo.")

# POSE_FRAMES relido do disco: mesma lista da celula 2/3, sem novo sentido.
# POSE_USADA vem da celula 3b: sao os frames preprocessados quando
# POSE_INPUT_MODE != render_direto, ou os originais caso contrario.
POSE_FRAMES = list(POSE_USADA)
for p in FRAMES_OUT.glob("*.png"):
    p.unlink()

COMFY_IN = pathlib.Path("/content/ComfyUI/input")
COMFY_IN.mkdir(parents=True, exist_ok=True)

def _para_comfy(origem, nome):
    destino = COMFY_IN / nome
    if pathlib.Path(origem).resolve() != destino.resolve():
        shutil.copy2(origem, destino)
    return nome

nome_chibi = _para_comfy(CHIBI, "pt_chibi" + CHIBI.suffix)
nome_ref = _para_comfy(REF_EXTRA, "pt_ref" + REF_EXTRA.suffix) if REF_EXTRA else None

GERACAO = []
t0 = time.time()
for i, pose in enumerate(POSE_FRAMES):
    seed_i = SEED if SEED_MODE == "fixa_em_todos_os_frames" else SEED + i
    nome_pose = _para_comfy(pose, f"pt_pose_{i:02d}.png")

    wf = json.loads(json.dumps(WF))
    wf["4"]["inputs"]["image"] = nome_pose      # POSE
    wf["13"]["inputs"]["image"] = nome_chibi    # IDENTIDADE
    if nome_ref:
        wf["16"]["inputs"]["image"] = nome_ref
    else:
        wf["16"]["inputs"]["image"] = nome_chibi
    wf["5"]["inputs"]["text"] = PROMPT
    ks = wf["10"]["inputs"]
    ks["seed"], ks["steps"], ks["cfg"] = seed_i, STEPS, float(CFG)
    for no in wf.values():
        if isinstance(no, dict) and no.get("class_type") == "EmptySD3LatentImage":
            w, h = Image.open(CHIBI).size
            no["inputs"]["width"], no["inputs"]["height"] = w, h
    wf["12"]["inputs"]["filename_prefix"] = f"pt_{ANIM}_{i:02d}"
    if USA_LORA:
        wf["20"]["inputs"]["lora_name"] = LORA_FILE
        wf["20"]["inputs"]["strength_model"] = float(LORA_STRENGTH)

    resolvido = {k: v for k, v in wf.items() if not k.startswith("_")}
    for no in resolvido.values():
        for campo, valor in no.get("inputs", {}).items():
            if isinstance(valor, str) and valor.startswith("%%"):
                raise SystemExit(
                    f"BLOCKED — placeholder {valor} nao resolvido no frame {i}.")

    req = urllib.request.Request(
        "http://127.0.0.1:8188/prompt",
        data=json.dumps({"prompt": resolvido}).encode(),
        headers={"Content-Type": "application/json"})
    pid = json.load(urllib.request.urlopen(req, timeout=30))["prompt_id"]

    saida = None
    for _ in range(360):
        time.sleep(2)
        with urllib.request.urlopen(
                f"http://127.0.0.1:8188/history/{pid}", timeout=10) as r:
            hist = json.load(r)
        if pid in hist:
            for no in hist[pid].get("outputs", {}).values():
                for img in no.get("images", []):
                    saida = img
            break
    if not saida:
        raise SystemExit(f"BLOCKED — frame {i} nao produziu imagem.")

    url = (f"http://127.0.0.1:8188/view?filename={saida['filename']}"
           f"&subfolder={saida.get('subfolder','')}&type={saida.get('type','output')}")
    destino = FRAMES_OUT / f"{ANIM}_{i:02d}.png"
    destino.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    GERACAO.append({"index": i, "pose_frame": pose.name, "seed": seed_i,
                    "output": destino.name})
    print(f"frame {i}/{len(POSE_FRAMES) - 1} ok  ({time.time() - t0:.0f}s)")

print()
print(f"{len(GERACAO)} frames em {time.time() - t0:.0f}s")


In [ ]:
#@title 8. Avaliacao: sheet, GIF em loop e escalas reais { display-mode: "form" }
OUT = sorted(FRAMES_OUT.glob(f"{ANIM}_*.png"))
if not OUT:
    raise SystemExit("BLOCKED — rode a celula 7.")

ims = [Image.open(p).convert("RGBA") for p in OUT]
fw, fh = ims[0].size
sheet = Image.new("RGBA", (fw * len(ims), fh), (0, 0, 0, 0))
for i, im in enumerate(ims):
    sheet.paste(im.resize((fw, fh)), (i * fw, 0))
SHEET_PATH = BASE / f"{CHARACTER_ID}_{ANIM}_{POSE_INPUT_MODE}_sheet.png"
sheet.save(SHEET_PATH)
print("sheet:", SHEET_PATH.name, sheet.size)

GIFS = []
if GERAR_GIF:
    for escala in ESCALAS:
        quadros = [im.resize((max(1, int(fw * escala / fh)), escala),
                             Image.NEAREST) for im in ims]
        g = BASE / f"{ANIM}_{POSE_INPUT_MODE}_loop_{escala}px.gif"
        quadros[0].save(g, save_all=True, append_images=quadros[1:],
                        duration=100, loop=0, disposal=2)
        GIFS.append(g)
        print("gif:", g.name)

# Diagnostico, NAO avaliacao de qualidade: quanto muda entre frames
# consecutivos. Numero baixo nao significa bom; numero alto sugere flicker.
import numpy as np
deltas = []
for a, b in zip(ims, ims[1:]):
    x = np.array(a.convert("RGB"), dtype=float)
    y = np.array(b.convert("RGB"), dtype=float)
    deltas.append(float(np.abs(x - y).mean()))
DELTA_MEDIO = sum(deltas) / len(deltas) if deltas else 0.0

print()
print("delta medio entre frames consecutivos: {:.2f}/255".format(DELTA_MEDIO))
print("  DIAGNOSTICO, nao qualidade. Mede quanto muda de um frame ao outro;")
print("  nao distingue movimento legitimo de cintilacao. Metrica boa != bom.")
print()
display(sheet)
for g in GIFS:
    print(g.name)
    display(Image.open(g))

print()
print("[HUMAN REVIEW REQUIRED]")
print("  Olhe o GIF de 64px em LOOP. A pergunta unica e:")
print("  a franja, os chifres, o contorno e as cores ficam ESTAVEIS,")
print("  ou cintilam de um frame para o outro?")
print("  Frames isolados sempre parecem bons — o defeito so aparece no loop.")
print("  Aprovar ou reprovar e decisao sua, nao do agente.")


In [ ]:
#@title 9. Empacotar { display-mode: "form" }
import zipfile, datetime, hashlib

# O modo entra no nome: comparar os 4 sem isso sobrescreveria os anteriores.
ZIP = pathlib.Path(
    f"/content/{CHARACTER_ID}_{ANIM}_{POSE_INPUT_MODE}_pose_transfer.zip")
RECIPE = {
    "experiment": "pose_transfer",
    "adr": "ADR-007",
    "status": "EXPERIMENTAL",
    "nota_adr_002": ("Estrategia (A) do ADR-002, descartada para producao por "
                     "drift/flicker. Este resultado NAO promove (A) a rota "
                     "oficial; rig cutout segue primario."),
    "character_id": CHARACTER_ID,
    "animation": ANIM,
    "frame_count": len(OUT),
    "created_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "prompt": PROMPT,
    "prompt_type": PROMPT_SOURCE,
    "character_specific_prompt": CHARACTER_SPECIFIC,
    "negative_custom": NEGATIVE_CUSTOM,
    "seed": SEED,
    "seed_mode": SEED_MODE,
    "steps": STEPS,
    "cfg": CFG,
    "pose_input_mode": POSE_INPUT_MODE,
    "preprocessor": PREPROC_INFO,
    "lora": ({"file": LORA_FILE, "strength": LORA_STRENGTH,
              "repo": "thedeoxen/refcontrol-FLUX.2-klein-4B-reference-depth-lora",
              "revision": "0ae1ef7f9acc4e55ec2237360943c3c3032d3583",
              "sha256": "65ec4c71fa7538b2201481928609a5836773f0dc06a4041b2d28abd05826c401",
              "license": "Apache-2.0",
              "commercial_status": "approved",
              "base_mismatch": ("LoRA declara klein-base-4B (nao destilada); "
                                "pipeline usa a destilada. [TEST REQUIRED]")}
             if USA_LORA else None),
    "workflow": str(WF_PATH.relative_to(REPO)),
    "workflow_sha256": hashlib.sha256(WF_PATH.read_bytes()).hexdigest(),
    "chibi_input": CHIBI.name,
    "chibi_sha256": CHIBI_SHA,
    "chibi_pixel_sha256": CHIBI_PIXEL_SHA,
    "ref_extra": REF_EXTRA.name if REF_EXTRA else None,
    "mannequin_mode": MANNEQUIN_MODE,
    "mannequin_source": "Blender render (Mixamo rig)",
    "mannequin_license": {
        "fonte": "Mixamo (Adobe)",
        "uso_comercial": True,
        "redistribuicao_de_arquivos_brutos": "proibida",
        "uso_para_treino_de_ml": "proibido",
        "nota": "Somente PNG renderizado e versionado. Nenhum treino ocorre."
    },
    "delta_medio_entre_frames": round(DELTA_MEDIO, 3),
    "delta_nota": ("DIAGNOSTICO, nao qualidade. Nao distingue movimento de "
                   "cintilacao."),
    "geracao": GERACAO,
    "approval_status": "experimental",
    "aprovacao": "[HUMAN REVIEW REQUIRED] — avaliar o GIF em loop a 64px",
}

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(FRAMES_OUT.glob("*.png")):
        z.write(p, f"frames/{p.name}")
    for p in sorted(FRAMES_IN.glob("*.png")):
        z.write(p, f"pose_input/{p.name}")
    z.write(SHEET_PATH, SHEET_PATH.name)
    for g in GIFS:
        z.write(g, f"loop/{g.name}")
    z.write(CHIBI, f"input/{CHIBI.name}")
    if REF_EXTRA:
        z.write(REF_EXTRA, f"input/{REF_EXTRA.name}")
    z.writestr("recipe.json", json.dumps(RECIPE, indent=2, ensure_ascii=False))
    z.writestr("workflow.resolved.json",
               json.dumps(resolvido, indent=2, ensure_ascii=False))
    z.writestr("RELATORIO.md", f"""# Pose transfer — {CHARACTER_ID} / {ANIM}

**EXPERIMENTAL (ADR-007).** Estrategia (A) do ADR-002, que foi descartada
para producao por drift e flicker. Este resultado nao a promove a rota
oficial.

| | |
|---|---|
| modo de pose | `{POSE_INPUT_MODE}` |
| frames | {len(OUT)} |
| seed | {SEED} ({SEED_MODE}) |
| steps / cfg | {STEPS} / {CFG} |
| prompt | `{PROMPT_SOURCE}` |
| character_specific_prompt | {CHARACTER_SPECIFIC} |
| delta medio entre frames | {DELTA_MEDIO:.2f}/255 |

O delta e **diagnostico**, nao medida de qualidade: nao separa movimento
legitimo de cintilacao.

## Como avaliar

Abra `loop/{ANIM}_{POSE_INPUT_MODE}_loop_64px.gif` e assista em loop. O defeito previsto pelo
ADR-002 e **cintilacao** e nao aparece em frames isolados.

`[HUMAN REVIEW REQUIRED]` — o agente nao julga o resultado.
""")

print("ZIP:", ZIP, f"({ZIP.stat().st_size / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(str(ZIP))
except Exception as e:
    print("[fora do Colab] baixe manualmente:", e)
